# 04 — Model Training & Comparison
**Home Credit Default Risk — Capstone Step 3**

Objective: train 3 model types, compare performance using financial metrics, and select the best model.

**Target: AUROC ≥ 0.72 on the held-out test set.**

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              average_precision_score, f1_score)
import joblib
import os

sns.set_style('whitegrid')
DATA_DIR = '../data/'

## 1. Load Data and Split

In [11]:
df = pd.read_csv(DATA_DIR + 'final_train.csv')
print(df.shape)

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Sauvegarde explicite du test set pour la démo/documentation
test_set = X_test.copy()
test_set['TARGET'] = y_test
test_set.to_csv('../data/test_set.csv', index=False)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train default rate:", y_train.mean().round(4), "Test default rate:", y_test.mean().round(4))

(307511, 230)
Train: (246008, 228) Test: (61503, 228)
Train default rate: 0.0807 Test default rate: 0.0807


## 2. Train Model 1: Logistic Regression (Baseline)

L2-regularized (Ridge, the sklearn default) — per Step 2's VIF findings, this handles the moderate multicollinearity in the engineered ratio features gracefully.

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)
print("Logistic Regression trained.")

Logistic Regression trained.


## 3. Train Model 2: Random Forest

In [13]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Random Forest trained.")

Random Forest trained.


## 4. Train Model 3: Gradient Boosting

In [14]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train, sample_weight=sample_weights)
print("Gradient Boosting trained.")

Gradient Boosting trained.


## 5. Financial Metrics for All Models

AUROC, Gini (2×AUROC−1), KS statistic, AUPRC, F1 at the optimal threshold — per the brief, accuracy is not used since it's misleading on this imbalanced dataset.

In [16]:
def ks_statistic(y_true, y_proba):
    """Kolmogorov-Smirnov statistic: max separation between the cumulative
    distributions of the positive and negative class."""
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    return np.max(np.abs(tpr - fpr))

def best_f1_threshold(y_true, y_proba):
    """Find the threshold that maximizes F1, rather than defaulting to 0.5."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1s[:-1])  # last point has no corresponding threshold
    return thresholds[best_idx], f1s[best_idx]

models = {
    'Logistic Regression': log_reg,
    'Random Forest': rf,
    'Gradient Boosting': gb,
}

results = []
proba_dict = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        proba = model.predict_proba(X_test)[:, 1]
    proba_dict[name] = proba

    auroc = roc_auc_score(y_test, proba)
    gini = 2 * auroc - 1
    ks = ks_statistic(y_test, proba)
    auprc = average_precision_score(y_test, proba)
    best_thresh, best_f1 = best_f1_threshold(y_test, proba)

    results.append({
        'Model': name,
        'AUROC': auroc,
        'Gini': gini,
        'KS': ks,
        'AUPRC': auprc,
        'F1 (optimal threshold)': best_f1,
        'Optimal threshold': best_thresh,
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df.round(4)


,AUROC,Gini,KS,AUPRC,F1 (optimal threshold),Optimal threshold
Model,,,,,,
Logistic Regression,0.7543,0.5087,0.3795,0.2411,0.3093,0.6755
Random Forest,0.7456,0.4912,0.3694,0.2257,0.2942,0.6002
Gradient Boosting,0.7653,0.5307,0.3948,0.2573,0.3192,0.6837


**Interpretation:**

Gradient Boosting wins on both AUROC (0.7653) and AUPRC (0.2573), as well as every other metric (Gini 0.5307, KS 0.3948, F1 0.3192).  

Logistic Regression comes second on AUROC (0.7543), narrowly beating Random Forest (0.7456), while on AUPRC Logistic Regression (0.2411) also edges out Random Forest (0.2257). 

All three models exceed the project's AUROC ≥ 0.72 target. The gap between Gradient Boosting and Logistic Regression is modest (~0.011 AUROC), worth noting in the final documentation as a trade-off between marginal performance gain and interpretability, even though Gradient Boosting is selected as the best model going forward.

The relatively narrow gap between Logistic Regression and Gradient Boosting suggests the engineered features (ratios, EXT_SOURCE combinations) capture much of the predictive signal in an already fairly linear/monotone form.

## 6. ROC and Precision-Recall Curves